# Demo 1: RAG Pipeline & MCP

Build a Retrieval-Augmented Generation pipeline that grounds LLM responses in actual clinical guidelines, then see how MCP standardizes tool integration.

## Learning Objectives

- Build a vector store from clinical documents using ChromaDB
- Implement the full RAG pipeline: embed → retrieve → augment → generate
- Compare RAG-grounded responses to direct LLM answers
- Understand how MCP standardizes tool discovery

## Setup

In [1]:
%pip install -q sentence-transformers chromadb openai python-dotenv mcp

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import numpy as np
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
import chromadb
from openai import OpenAI

load_dotenv()

# Configure client — works with OpenRouter or OpenAI
if os.environ.get("OPENROUTER_API_KEY"):
    client = OpenAI(
        api_key=os.environ["OPENROUTER_API_KEY"],
        base_url="https://openrouter.ai/api/v1",
    )
    MODEL = "openai/gpt-4o-mini"
elif os.environ.get("OPENAI_API_KEY"):
    client = OpenAI()
    MODEL = "gpt-4o-mini"
else:
    raise ValueError("Set OPENROUTER_API_KEY or OPENAI_API_KEY in .env")

print(f"Using model: {MODEL}")

Using model: openai/gpt-4o-mini


## Section 1: Clinical Knowledge Base

We'll work with synthetic clinical guideline chunks — the kind of documents a hospital might want an LLM to reference when answering clinical questions.

In [3]:
clinical_knowledge = [
    {
        "id": "hypertension_1",
        "text": "Stage 1 hypertension is defined as systolic blood pressure 130-139 mmHg or diastolic 80-89 mmHg. Initial treatment includes lifestyle modifications: weight loss, DASH diet, sodium restriction (<2300mg/day), regular exercise (150 min/week moderate intensity).",
        "source": "JNC8 Guidelines"
    },
    {
        "id": "hypertension_2",
        "text": "First-line antihypertensive medications include thiazide diuretics, ACE inhibitors, ARBs, and calcium channel blockers. Choice depends on comorbidities: ACE inhibitors or ARBs preferred in diabetes and chronic kidney disease.",
        "source": "JNC8 Guidelines"
    },
    {
        "id": "diabetes_1",
        "text": "Type 2 diabetes is diagnosed with fasting glucose ≥126 mg/dL, HbA1c ≥6.5%, or 2-hour glucose ≥200 mg/dL during OGTT. Metformin is first-line therapy unless contraindicated. Target HbA1c <7% for most adults.",
        "source": "ADA Standards of Care"
    },
    {
        "id": "diabetes_2",
        "text": "For patients with type 2 diabetes and cardiovascular disease, SGLT2 inhibitors or GLP-1 receptor agonists with proven cardiovascular benefit are recommended regardless of HbA1c. Examples: empagliflozin, liraglutide.",
        "source": "ADA Standards of Care"
    },
    {
        "id": "chest_pain_1",
        "text": "Acute chest pain evaluation: HEART score assesses History, ECG, Age, Risk factors, and Troponin. Score 0-3 is low risk (discharge with follow-up), 4-6 is intermediate (admit for observation), 7-10 is high risk (early invasive strategy).",
        "source": "AHA Chest Pain Guidelines"
    },
    {
        "id": "chest_pain_2",
        "text": "STEMI diagnosis requires ST elevation ≥1mm in 2 contiguous leads or new LBBB with symptoms. Door-to-balloon time goal <90 minutes for primary PCI. If PCI not available within 120 minutes, fibrinolysis within 30 minutes.",
        "source": "AHA Chest Pain Guidelines"
    },
]

print(f"{len(clinical_knowledge)} guideline chunks loaded")
for doc in clinical_knowledge:
    print(f"  [{doc['source']}] {doc['text'][:60]}...")

6 guideline chunks loaded
  [JNC8 Guidelines] Stage 1 hypertension is defined as systolic blood pressure 1...
  [JNC8 Guidelines] First-line antihypertensive medications include thiazide diu...
  [ADA Standards of Care] Type 2 diabetes is diagnosed with fasting glucose ≥126 mg/dL...
  [ADA Standards of Care] For patients with type 2 diabetes and cardiovascular disease...
  [AHA Chest Pain Guidelines] Acute chest pain evaluation: HEART score assesses History, E...
  [AHA Chest Pain Guidelines] STEMI diagnosis requires ST elevation ≥1mm in 2 contiguous l...


## Section 2: Embedding & ChromaDB Indexing

ChromaDB is an in-memory vector database. We encode each chunk into an embedding vector, then store them for fast similarity search.

In [4]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

chroma_client = chromadb.Client()
collection = chroma_client.create_collection(
    name="clinical_guidelines",
    metadata={"description": "Clinical practice guidelines"},
)

documents = [doc["text"] for doc in clinical_knowledge]
ids = [doc["id"] for doc in clinical_knowledge]
metadatas = [{"source": doc["source"]} for doc in clinical_knowledge]
embeddings = embedding_model.encode(documents).tolist()

collection.add(
    documents=documents,
    ids=ids,
    metadatas=metadatas,
    embeddings=embeddings,
)

print(f"Indexed {collection.count()} chunks in ChromaDB")
print(f"Embedding dimension: {len(embeddings[0])}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Indexed 6 chunks in ChromaDB
Embedding dimension: 384


## Section 3: RAG Query Function

The core RAG loop: embed the question → retrieve similar chunks → inject them as context → generate a grounded answer.

In [5]:
def rag_query(question, n_results=3, show_sources=True):
    """Answer a question using retrieved guideline chunks as context."""
    # Step 1: Retrieve
    query_embedding = embedding_model.encode([question]).tolist()
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=n_results,
        include=["documents", "metadatas", "distances"],
    )

    if show_sources:
        print("Retrieved documents:")
        for i, (doc, meta, dist) in enumerate(zip(
            results["documents"][0],
            results["metadatas"][0],
            results["distances"][0],
        )):
            print(f"  {i+1}. [{meta['source']}] (distance: {dist:.3f})")
            print(f"     {doc[:80]}...")
        print()

    # Step 2: Augment — build context from retrieved chunks
    context = "\n\n".join(results["documents"][0])

    # Step 3: Generate
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a clinical assistant. Answer based ONLY on the provided context. "
                    "If the context doesn't contain enough information, say so. "
                    "Cite the guideline source when possible."
                ),
            },
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
        ],
        temperature=0,
    )
    return response.choices[0].message.content

In [6]:
questions = [
    "What is the first-line treatment for hypertension in a patient with diabetes?",
    "How do I evaluate a patient with acute chest pain?",
    "What HbA1c level indicates diabetes?",
]

for q in questions:
    print("=" * 60)
    print(f"Q: {q}\n")
    answer = rag_query(q)
    print(f"A: {answer}\n")

Q: What is the first-line treatment for hypertension in a patient with diabetes?

Retrieved documents:
  1. [JNC8 Guidelines] (distance: 0.631)
     First-line antihypertensive medications include thiazide diuretics, ACE inhibito...
  2. [JNC8 Guidelines] (distance: 0.927)
     Stage 1 hypertension is defined as systolic blood pressure 130-139 mmHg or diast...
  3. [ADA Standards of Care] (distance: 0.936)
     For patients with type 2 diabetes and cardiovascular disease, SGLT2 inhibitors o...



A: The first-line treatment for hypertension in a patient with diabetes includes ACE inhibitors or ARBs, as they are preferred in this population.

Q: How do I evaluate a patient with acute chest pain?

Retrieved documents:
  1. [AHA Chest Pain Guidelines] (distance: 0.645)
     Acute chest pain evaluation: HEART score assesses History, ECG, Age, Risk factor...
  2. [AHA Chest Pain Guidelines] (distance: 1.489)
     STEMI diagnosis requires ST elevation ≥1mm in 2 contiguous leads or new LBBB wit...
  3. [JNC8 Guidelines] (distance: 1.736)
     First-line antihypertensive medications include thiazide diuretics, ACE inhibito...



A: To evaluate a patient with acute chest pain, you can use the HEART score, which assesses the following components:

1. **History**: Evaluate the patient's history of chest pain.
2. **ECG**: Perform an electrocardiogram to check for abnormalities.
3. **Age**: Consider the patient's age as a risk factor.
4. **Risk factors**: Assess for other risk factors such as hypertension, diabetes, smoking, etc.
5. **Troponin**: Measure troponin levels to check for myocardial injury.

Based on the HEART score:
- A score of **0-3** indicates low risk, and the patient can be discharged with follow-up.
- A score of **4-6** indicates intermediate risk, and the patient should be admitted for observation.
- A score of **7-10** indicates high risk, and an early invasive strategy should be considered.

Additionally, if there is a suspicion of STEMI, look for ST elevation of ≥1mm in 2 contiguous leads or a new left bundle branch block (LBBB) with symptoms. The goal for door-to-balloon time for primary PCI 

A: An HbA1c level of ≥6.5% indicates diabetes.



## Section 4: RAG vs Direct LLM

What happens when the model answers *without* retrieved context? For well-known clinical facts (like hypertension thresholds) the LLM may already know the answer — but for organization-specific protocols, recent guideline updates, or internal policy, the model has no choice but to guess or refuse.

In [7]:
def direct_llm_query(question):
    """Ask the LLM directly — no retrieval."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You are a clinical assistant. Be concise."},
            {"role": "user", "content": question},
        ],
        temperature=0,
    )
    return response.choices[0].message.content


# Use a question where the guideline-specific detail matters:
# The HEART score thresholds (exact cutoffs) vary by institutional protocol.
# Our guidelines specify 0-3/4-6/7-10. The LLM may give different cutoffs.
test_q = (
    "According to the AHA Chest Pain Guidelines, what HEART score threshold "
    "separates low-risk from intermediate-risk patients, and what is the recommended "
    "action for each risk category?"
)

print("RAG Response (grounded in our guidelines):")
rag_answer = rag_query(test_q)
print(rag_answer)
print("\n" + "-" * 40 + "\n")
print("Direct LLM Response (from training data — may differ or hallucinate thresholds):")
print(direct_llm_query(test_q))

RAG Response (grounded in our guidelines):
Retrieved documents:
  1. [AHA Chest Pain Guidelines] (distance: 0.488)
     Acute chest pain evaluation: HEART score assesses History, ECG, Age, Risk factor...
  2. [JNC8 Guidelines] (distance: 1.290)
     Stage 1 hypertension is defined as systolic blood pressure 130-139 mmHg or diast...
  3. [AHA Chest Pain Guidelines] (distance: 1.509)
     STEMI diagnosis requires ST elevation ≥1mm in 2 contiguous leads or new LBBB wit...



According to the AHA Chest Pain Guidelines, a HEART score of 0-3 separates low-risk patients from those at intermediate risk (score of 4-6). The recommended action for low-risk patients is to discharge them with follow-up, while for intermediate-risk patients, the recommendation is to admit them for observation.

----------------------------------------

Direct LLM Response (from training data — may differ or hallucinate thresholds):


According to the AHA Chest Pain Guidelines, a HEART score of 0-3 indicates low risk, while a score of 4-6 indicates intermediate risk. 

- **Low-risk (HEART score 0-3)**: Recommended action is to consider discharge with appropriate follow-up and reassurance.
- **Intermediate-risk (HEART score 4-6)**: Recommended action is to consider further diagnostic testing and observation.


## Section 5: RAG with Citations

Number the source chunks so the model can reference them by number — makes it easy to trace which guideline supports each claim.

In [8]:
def rag_with_citations(question, n_results=3):
    """RAG that returns answer + numbered source citations."""
    query_embedding = embedding_model.encode([question]).tolist()
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=n_results,
        include=["documents", "metadatas"],
    )

    context_parts = []
    sources = []
    for i, (doc, meta) in enumerate(
        zip(results["documents"][0], results["metadatas"][0])
    ):
        context_parts.append(f"[{i+1}] {doc}")
        sources.append(f"[{i+1}] {meta['source']}")

    context = "\n\n".join(context_parts)

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "Answer based on the numbered sources. "
                    "Include citation numbers [1], [2], etc."
                ),
            },
            {"role": "user", "content": f"Sources:\n{context}\n\nQuestion: {question}"},
        ],
        temperature=0,
    )

    return {"answer": response.choices[0].message.content, "sources": sources}


result = rag_with_citations(
    "What medications are recommended for diabetic patients with heart disease?"
)
print(f"Answer: {result['answer']}\n")
print("Sources:")
for s in result["sources"]:
    print(f"  {s}")

Answer: For patients with type 2 diabetes who also have cardiovascular disease, it is recommended to use SGLT2 inhibitors or GLP-1 receptor agonists that have proven cardiovascular benefits, regardless of their HbA1c levels. Examples of these medications include empagliflozin and liraglutide [1].

Sources:
  [1] ADA Standards of Care
  [2] JNC8 Guidelines
  [3] ADA Standards of Care


## Section 6: MCP — Standardized Tool Discovery

We built our RAG pipeline by hand: custom embedding code, custom ChromaDB queries, custom prompt assembly. **Model Context Protocol (MCP)** standardizes all of this — pre-built servers expose tools that any LLM client can discover and call.

The pattern: connect to an MCP server → discover available tools → convert to OpenAI format → use with our existing agent code.

In [9]:
# MCP tool discovery uses async — focus on the pattern, not the async details

import asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client


async def explore_mcp_server():
    """Connect to an MCP server and list available tools."""
    server = StdioServerParameters(
        command="npx",
        args=["-y", "@modelcontextprotocol/server-filesystem", "."],
    )

    async with stdio_client(server) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            tools = await session.list_tools()
            print("Available MCP Tools:")
            print("-" * 40)
            for tool in tools.tools:
                print(f"  {tool.name}: {tool.description[:60]}...")

            return tools


# Uncomment to run (requires Node.js/npx installed):
# tools = asyncio.run(explore_mcp_server())

In [10]:
def mcp_to_openai_tools(mcp_tools):
    """Convert MCP tool definitions to OpenAI function calling format.

    Once converted, these work with any agent loop that accepts OpenAI-format tools.
    """
    return [
        {
            "type": "function",
            "function": {
                "name": tool.name,
                "description": tool.description,
                "parameters": tool.inputSchema,
            },
        }
        for tool in mcp_tools.tools
    ]


# After conversion, plug into an agent loop:
# openai_tools = mcp_to_openai_tools(tools)
# response = client.chat.completions.create(model=MODEL, tools=openai_tools, ...)

**Key insight**: The agent loop stays the same — MCP just standardizes how tools are discovered and called. What we built by hand in this demo is exactly what MCP automates.

| Manual Approach (this demo) | MCP Approach |
|:---|:---|
| Define each tool function yourself | Use pre-built servers |
| Wire up tool execution manually | Standard call/response protocol |
| Build each integration from scratch | Plug-and-play servers |
| Great for learning | Great for production |

## Exercises

1. **Add more guidelines**: Add chunks about a new topic (e.g., anticoagulation, asthma management) and test retrieval
2. **Chunking experiment**: Split the longer guidelines into smaller chunks — does retrieval quality change?
3. **Hybrid search**: Filter by source metadata before doing similarity search
4. **Evaluation**: Write 5 questions with known answers and check if RAG gets them right

## Key Takeaways

- RAG grounds LLM responses in retrieved documents, reducing hallucination
- Vector databases (ChromaDB) enable fast semantic retrieval via embeddings
- Citation tracking makes RAG outputs verifiable
- MCP standardizes tool discovery so you don't have to wire everything by hand